In [1]:
# =========================================================
# Task04 改3: 箱を到達圏内 / ボール追従 / 収まり可視化
# 出力: /content/task04_sort4.mp4
# =========================================================
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.patches import Circle, Rectangle
import os

FPS=15; DURATION=75.0; NFRAMES=int(FPS*DURATION)
rng=np.random.default_rng(0)
BASE=(0.0,0.2); L1,L2=0.8,0.6
REACH=L1+L2-0.05

# ★ 箱を到達圏内に（x=±1.05, 箱の幅0.4）
BINS={"red":(-1.05,'C3'),"blue":(1.05,'C0')}
BOX_W=0.4; BOX_H=0.35; BOX_Y=0.0
PICK_R=0.20
N_OBJ=20
MOVE_SPEED=0.9
PICK_DWELL=0.5
PLACE_DWELL=0.5

def ik_2link(target, base=BASE, L1=L1, L2=L2):
    dx,dy=target[0]-base[0],target[1]-base[1]
    d=np.hypot(dx,dy); d=min(d,L1+L2-1e-3); d=max(d,abs(L1-L2)+1e-3)
    a=np.arctan2(dy,dx)
    cos1=np.clip((L1**2+d**2-L2**2)/(2*L1*d),-1,1); th1=a-np.arccos(cos1)
    cos2=np.clip((L1**2+L2**2-d**2)/(2*L1*L2),-1,1); th2=np.pi-np.arccos(cos2)
    return th1,th2

def fk_2link(th1,th2,base=BASE,L1=L1,L2=L2):
    x1=base[0]+L1*np.cos(th1); y1=base[1]+L1*np.sin(th1)
    x2=x1+L2*np.cos(th1+th2);  y2=y1+L2*np.sin(th1+th2)
    return np.array([base,(x1,y1),(x2,y2)])

class MockDetector:
    def detect(self, objs):
        return [{"color":o["color"],"box":(o["x"],o["y"]),"conf":0.9} for o in objs]
detector=MockDetector()

fig,ax=plt.subplots(figsize=(6.4,4.8),dpi=100)
ax.set_xlim(-3,3); ax.set_ylim(-0.5,3); ax.set_aspect('equal'); ax.grid(alpha=0.2)
ax.add_patch(Rectangle((-3,-0.1),6,0.1,color='0.6'))
ax.plot([BASE[0]],[BASE[1]],'s',color='k',ms=6)

# ★ 箱を深めに描画（ボールが収まるのが見える）
for name,(x,c) in BINS.items():
    ax.add_patch(Rectangle((x-BOX_W/2,BOX_Y),BOX_W,BOX_H,
                 facecolor=c,alpha=0.25,edgecolor=c,lw=2))
    ax.text(x,BOX_Y+BOX_H+0.08,name,ha='center',fontsize=9,color=c)

arm_line,=ax.plot([],[],'-o',color='C0',lw=3,ms=5)
status=ax.text(-2.8,2.75,"",fontsize=9,color='C3')
count_txt=ax.text(1.6,2.75,"sorted: 0",fontsize=10,color='C2',fontweight='bold')
ax.set_title("Task04: YOLO Sort (reachable bins)",fontsize=10)

# ★ 物体は到達圏内（x:±0.9, y:0.8〜1.4）
def make_obj():
    color=rng.choice(["red","blue"])
    x=rng.uniform(-0.9,0.9); y=rng.uniform(0.8,1.4)
    return {"color":color,"x":x,"y":y,"patch":ax.add_patch(Circle((x,y),0.09,color=color))}
objs=[make_obj() for _ in range(N_OBJ)]

T=np.arange(NFRAMES)/FPS
HOME=(0.0,1.0)
state={"phase":"idle","target":None,"sorted":0,
       "tip":np.array(HOME,dtype=float),
       "dwell_until":0.0,"next_t":0.0,"pick_deadline":0.0}

def move_tip_toward(goal, speed):
    d=goal-state["tip"]; dist=np.linalg.norm(d)
    if dist<1e-3: return True
    step=min(dist, speed/FPS); state["tip"]=state["tip"]+d/dist*step
    return False

def animate(i):
    t=T[i]
    if state["phase"]=="idle" and objs and t>=state["next_t"]:
        o=objs[0]; state["target"]=o; state["phase"]="pick"
        state["pick_deadline"]=t+3.0

    if state["phase"]=="pick" and state["target"]:
        o=state["target"]; goal=np.array([o["x"],o["y"]])
        reached=move_tip_toward(goal, MOVE_SPEED)
        if reached or np.linalg.norm(state["tip"]-goal)<PICK_R:
            state["phase"]="pick_dwell"; state["dwell_until"]=t+PICK_DWELL
        elif t>state["pick_deadline"]:
            o["patch"].remove(); objs.remove(o)
            state["target"]=None; state["phase"]="idle"; state["next_t"]=t+0.3
        status.set_text("PICK")

    elif state["phase"]=="pick_dwell" and state["target"]:
        o=state["target"]
        # ★ ボールを先端に追従（くっついて運ぶ）
        o["patch"].center=(state["tip"][0],state["tip"][1])
        if t>=state["dwell_until"]:
            bx=BINS[o["color"]][0]
            state["phase"]="carry"; status.set_text("CARRY")
        else:
            status.set_text("GRAB")

    elif state["phase"]=="carry" and state["target"]:
        o=state["target"]; bx=BINS[o["color"]][0]
        # 箱の真上へ（箱の中に落とせる高さ）
        goal=np.array([bx, BOX_Y+BOX_H+0.15])
        reached=move_tip_toward(goal, MOVE_SPEED)
        # ★ 運搬中もボールを先端に追従
        o["patch"].center=(state["tip"][0],state["tip"][1])
        if reached or np.linalg.norm(state["tip"]-goal)<0.1:
            state["phase"]="place"; state["dwell_until"]=t+PLACE_DWELL

    elif state["phase"]=="place" and state["target"]:
        o=state["target"]; bx=BINS[o["color"]][0]
        if t>=state["dwell_until"]:
            # ★ 箱の中に収める（箱の底付近に固定）
            o["x"],o["y"]=bx, BOX_Y+BOX_H/2
            o["patch"].center=(o["x"],o["y"])
            o["patch"].set_radius(0.07)
            state["sorted"]+=1; count_txt.set_text(f"sorted: {state['sorted']}")
            objs.remove(o); state["target"]=None
            state["phase"]="idle"; state["next_t"]=t+0.3

    else:
        move_tip_toward(np.array(HOME), MOVE_SPEED)
        status.set_text("idle")

    th1,th2=ik_2link(tuple(state["tip"])); j=fk_2link(th1,th2)
    arm_line.set_data(j[:,0],j[:,1])
    return arm_line,status,count_txt

anim=FuncAnimation(fig,animate,frames=NFRAMES,interval=1000/FPS,blit=False)
out="/content/task04_sort4.mp4"
anim.save(out,writer=FFMpegWriter(fps=FPS,bitrate=1200),dpi=100)
plt.close(fig); print("saved:",out,f"{os.path.getsize(out)/1e6:.2f} MB")
from IPython.display import Video; Video(out,embed=True,width=640)

saved: /content/task04_sort4.mp4 1.37 MB
